In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
#dbutils.widgets.text('destination_schema_zone', 'None', '07-Destination Schema Zone') # change for testing
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')
#destination_schema_zone = dbutils.widgets.get('destination_schema_zone')    # change for testing

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_expenses_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) : # change for testing
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #      "ExpensesId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "ExpensesKey string,"
        #      "PartyKey string,"
        #      "CDI string,"
        #      "ExposureKey string,"
        #      "AgreementKey string,"
        #      "LegalActionKey string,"
        #      "InsuranceKey string,"
        #      "ExpensesTypeCode string,"
        #      "ExpensesType string,"
        #      "ExpensesDate timestamp,"
        #      "TotalAmount decimal(20,6),"
        #      "NetAmount decimal(20,6),"
        #      "TaxAmount decimal(19,4),"
        #      "VatAmount decimal(20,6),"
        #      "InternalCost decimal(19,10),"
        #      "OtherExpenses decimal(19,10),"
        #      "ExpensesCategoryCode string,"
        #      "ExpensesCategory string,"
        #      "ExpensesStatusCode string,"
        #      "ExpensesStatus string,"
        #      "VATPercentage string,"
        #      "PacketCode int,"
        #      "Packet string,"
        #      "Comments string,"
        #      "LawyerKey int,"
        #      "ExternalCompanyPartyKey int,"
        #      "InvoiceDate timestamp,"
        #      "InvoiceNumber string,"
        #      "InvoiceApprovalStatusCode int,"
        #      "InvoiceApprovalStatus int,"
        #      "ExpensesValidationStatusCode int,"
        #      "ExpensesValidationStatus int,"
        #      "ExpensesApprovalStatusCode int,"
        #      "ExpensesApprovalStatus int,"
        #      "BusinessDate date"
        #     )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Expenses" # θα γράψει σε αυτόν τον πίνακα # change for testing

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits", "Arotron_dec_flag"] # Legal Actions QC Decommission to Arotron - notebooks run

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

         # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
#


        #portfolio_arotron_dec = portfolio.where(col("Arotron_dec_flag") == 1)
        #portfolio_list = [row["PortfolioName"] for row in portfolio_arotron_dec.select("PortfolioName").collect()]
#
#
        ## ================== Delete From Delta Table ==================
        #if spark.catalog.tableExists(curated_table_path) is True:
        #    delete_from_curated_table_nj(
        #        spark,
        #        curated_table_path,
        #        run_date,  # type: ignore
        #        "arotron",
        #        portfolio_list
        #    )        
        #else:
        #    logging.info("Tables does not exist, no need to delete entries for business date!")
#
        # ============================== ArmFctActionExpenses ====================
        
        armfct_action_expenses_extra_filters = None
        if armfct_action_expenses_extra_filters is not None:
            armfct_action_expenses_extra_filters.append(date_filter)
            armfct_action_expenses_final_filters = " AND ".join(armfct_action_expenses_extra_filters)
        else:
            armfct_action_expenses_final_filters = date_filter

        armfct_action_expenses_selected_fields = [
            "ActionID",
            "ExpenseID",
            "ExpenseRefNo",
            "ExpenseInvoiceDate",
            "ExpenseStatusCode",
            "ExpenseAmount",
            "ExpenseVATAmount",
            "ExpenseVATCategoryName",
            "ExpensePayableAmount",
            "ExpenseNotes",
            "ExpenseTaxAmount",
            "ExpenseInvoiceDate",
            "ExpenseStatusName",
            "ExpenseTypeName",
            "ExpenseTypeCode",
            "Businessdate",
        ]

        armfct_action_expenses = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_armfctactionexpenses",
            armfct_action_expenses_final_filters,
            armfct_action_expenses_selected_fields,
        )


        # ============================== ArmFctLegalAction ====================

        armfct_legal_action_extra_filters = None
        if armfct_legal_action_extra_filters is not None:
            armfct_legal_action_extra_filters.append(date_filter)
            armfct_legal_action_final_filters = " AND ".join(armfct_legal_action_extra_filters)
        else:
            armfct_legal_action_final_filters = date_filter

        armfct_legal_action_selected_fields = [
            "ActionAssigneeID",
            "ActionAssigneeTeam",
            "ActionID",
            "ProposedByID",
            "CaseBEID",
            "Businessdate",
        ]

        armfct_legal_action = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_armfctlegalaction",
            armfct_legal_action_final_filters,
            armfct_legal_action_selected_fields,
        )
         
        # ============================== ArmDimAccount ====================

        arm_dim_account_extra_filters = None
        #arm_dim_account_extra_filters = []

        #last_record_indicator_filter = "LastRecordIndicator = 1"
        #arm_dim_account_extra_filters.append(last_record_indicator_filter)
        #arm_dim_account_extra_filters.append(expense_id_filter)
        if arm_dim_account_extra_filters is not None:
            arm_dim_account_extra_filters.append(date_filter)
            arm_dim_account_final_filters = " AND ".join(arm_dim_account_extra_filters)
        else:
           arm_dim_account_final_filters = date_filter

        arm_dim_account_selected_fields = [
            "CustomerNo",
            "AccountNo",
            "SecurityLevelCode",
            "CaseBEID",
            "LastRecordIndicator",
            "Businessdate",
        ]

        arm_dim_account = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotronmis_armdimaccount",
            arm_dim_account_final_filters,
            arm_dim_account_selected_fields,
        )

        # ============================== CrwOwner ====================

        crw_owner_extra_filters = None
        if crw_owner_extra_filters is not None:
            crw_owner_extra_filters.append(date_filter)
            crw_owner_final_filters = " AND ".join(crw_owner_extra_filters)
        else:
           crw_owner_final_filters = date_filter
        crw_owner_selected_fields = [
            "crwOwner_ID",
            "crwOwner_UserName",
            "Businessdate",
        ]

        crw_owner = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crwowner",
            crw_owner_final_filters,
            crw_owner_selected_fields,
        )

        # ============================== crmMLT ====================

        crm_mlt_extra_filters = None
        if crm_mlt_extra_filters is not None:
            crm_mlt_extra_filters.append(date_filter)
            crm_mlt_final_filters = " AND ".join(crm_mlt_extra_filters)
        else:
           crm_mlt_final_filters = date_filter

        crm_mlt_selected_fields = [
            "crmMLT_ID",
            "crmMLT_TextInLanguage",
            "crmMLT_LanguageID",
            "Businessdate",
        ]

        crm_mlt = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmMLT",
            crm_mlt_final_filters,
            crm_mlt_selected_fields,
        )

        # ============================== crmLOV ====================

        crm_lov_extra_filters = None
        if crm_lov_extra_filters is not None:
            crm_lov_extra_filters.append(date_filter)
            crm_lov_final_filters = " AND ".join(crm_lov_extra_filters)
        else:
           crm_lov_final_filters = date_filter

        crm_lov_selected_fields = [
            "crmLOV_ID",
            "crmLOV_ParentID",
            "crmLOV_GlobalAbbrCode",
            "Businessdate",
        ]

        crm_lov = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_crmlov",
            crm_lov_final_filters,
            crm_lov_selected_fields,
        )


        # ====================== batr_mlt  =========================
        # Batr_MLT equivalent (subquery)
        join_cond_batr_mlt = crm_lov["crmLOV_ID"] == crm_mlt["crmMLT_ID"]

        batr_mlt = (
            crm_lov.filter("crmLOV_ParentID = 'SYS_BATR_TYPE'")
            .join(
                crm_mlt.filter("crmMLT_LanguageID = 'SYS_LANG_GR'"),
                join_cond_batr_mlt,
                how="inner"
            )
            .select(
                crm_lov["crmLOV_ID"].alias("crmLOV_ID"),
                crm_mlt["crmMLT_TextInLanguage"].alias("crmMLT_TextInLanguage"),
                crm_lov["crmLOV_GlobalAbbrCode"].alias("crmLOV_GlobalAbbrCode")
            )
            .alias("Batr_MLT")
        )

        # ====================== Join Conditions ========================
        # Join conditions
        join_cond_legal_action = armfct_action_expenses["ActionID"] == armfct_legal_action["ActionID"]
        join_cond_dim_account = armfct_legal_action["CaseBEID"] == arm_dim_account["CaseBEID"]
        join_cond_crw_owner = armfct_legal_action["ProposedByID"] == crw_owner["crwOwner_ID"]
        join_cond_exp_type = ((armfct_action_expenses["ExpenseTypeName"] == batr_mlt["crmMLT_TextInLanguage"])  &
        (batr_mlt["crmLOV_GlobalAbbrCode"]==armfct_action_expenses["ExpenseTypeCode"]))
        join_cond_portfolio = (substring(arm_dim_account["CustomerNo"], 1, 3) == portfolio["prefix"])

        expenses_df = (
            armfct_action_expenses.alias("ae")
            .join(armfct_legal_action.alias("la"), join_cond_legal_action, how="left")
            .join(arm_dim_account.alias("acc"), join_cond_dim_account, how="left")
            .join(crw_owner.alias("co"), join_cond_crw_owner, how="left")
            .join(batr_mlt.alias("Batr_MLT"), join_cond_exp_type, how="left")
            .join(portfolio.alias("portfolio"), join_cond_portfolio, how="left")
            .select(
                # co (CrwOwner)
                crw_owner["crwOwner_UserName"].alias("LawyerUserID"),
                # acc (ArmDimAccount)
                arm_dim_account["CustomerNo"].alias("CustCDI"),
                arm_dim_account["AccountNo"].alias("ContNum"),
                arm_dim_account["SecurityLevelCode"].alias("PorfCode"),
                arm_dim_account["LastRecordIndicator"].alias("LastRecordIndicator"),
                #batr_mlt
                batr_mlt["crmLOV_ID"].alias("ExpCatID"),
                batr_mlt["crmLOV_GlobalAbbrCode"].alias("ExpTypeCode"),
                batr_mlt["crmMLT_TextInLanguage"].alias("ExpType"),
                # ae (ArmFctActionExpenses)
                armfct_action_expenses["ActionID"].alias("LegActID"),
                armfct_action_expenses["ExpenseID"].alias("ExpID"),
                armfct_action_expenses["ExpenseRefNo"].alias("InvNumber"),
                armfct_action_expenses["ExpenseInvoiceDate"].alias("ExpDate"),
                armfct_action_expenses["ExpenseStatusCode"].alias("ExpStatusID"),
                armfct_action_expenses["ExpenseAmount"].alias("ExpNetAmt"),
                armfct_action_expenses["ExpenseVATAmount"].alias("VATAmt"),
                armfct_action_expenses["ExpenseVATCategoryName"].alias("VATPs"),
                armfct_action_expenses["ExpensePayableAmount"].alias("PmtAmt"),
                armfct_action_expenses["ExpenseNotes"].alias("ExpComments"),
                current_timestamp().alias("ExportDT"),
                armfct_action_expenses["ExpenseTaxAmount"].alias("TaxAmount"),
                armfct_action_expenses["ExpenseInvoiceDate"].alias("InvoiceDate"),
                armfct_action_expenses["ExpenseStatusName"].alias("ExpenseStatusName"),
                armfct_action_expenses["Businessdate"].alias("Businessdate"),
                # la (ArmFctLegalAction)
                armfct_legal_action["ActionAssigneeID"].alias("LawyerKey"),
                armfct_legal_action["ActionAssigneeTeam"].alias("ExternalCompanyPartyKey"),
                # AccountNo as ExposureKey -- we did not keep this
                arm_dim_account["AccountNo"].alias("ExposureKey"),
                #portfolio
                portfolio["unit_code_profits"],
                portfolio["PortfolioName"]
                #portfolio["Arotron_dec_flag"] # Legal Actions QC Decommission to Arotron - notebooks run
            )
        )

        # Application of filter LastRecordIndicator = 1
        expenses_df = expenses_df.filter("LastRecordIndicator = 1")

        # ================= Set the Keys for Hashkey================

        ExpenseId__primary_key_fields = [
            expenses_df.ExpID,
            lit("arotron"),
        ]

        # =============== Creation of Hashkeys ===================

        target_table_arotron = expenses_df.withColumn(
            "ExpensesId",
            sha2(
                concat_ws(";", *ExpenseId__primary_key_fields),
                256,
            ),
        ).withColumn(
            "AgreementKey",
            when(expenses_df.ContNum.isNull(), lit(None)).otherwise(
                expenses_df.ContNum.cast("string")
            ),
        ).distinct()

        # =========== Selection of Final Columns ================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.ExpensesId.cast("string").alias("ExpensesId"),
            lit("arotron").cast("string").alias("SourceSystem"),
            coalesce(portfolio["unit_code_profits"], lit("-9998"))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio["PortfolioName"], lit("Core Wholesale"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            target_table_arotron.ExpID.cast("string").alias("ExpensesKey"),
            target_table_arotron.CustCDI.cast("string").alias("PartyKey"),
            target_table_arotron.CustCDI.cast("string").alias("CDI"),
            lit(None).cast("string").alias("ExposureKey"), # IT has given us here the AgreementKey. We had this as null         anyway.
            target_table_arotron.AgreementKey.cast("string").alias("AgreementKey"),
            target_table_arotron.LegActID.cast("string").alias("LegalActionKey"),
            lit(None).cast("string").alias("InsuranceKey"),
            target_table_arotron.ExpType.cast("string").alias("ExpensesTypeCode"), #Probably ExpTypeCode but we had it as       the same with 'ExpensesType' so we keep it the same
            target_table_arotron.ExpType.cast("string").alias("ExpensesType"),     
            target_table_arotron.ExpDate.cast("timestamp").alias("ExpensesDate"),
            target_table_arotron.PmtAmt.cast("decimal(20,6)").alias("TotalAmount"),
            target_table_arotron.ExpNetAmt.cast("decimal(20,6)").alias("NetAmount"),
            target_table_arotron.TaxAmount.cast("decimal(19,4)").alias("TaxAmount"),
            target_table_arotron.VATAmt.cast("decimal(20,6)").alias("VatAmount"),
            lit(None).cast("decimal(19,10)").alias("InternalCost"),
            lit(None).alias("OtherExpenses"),
            target_table_arotron.ExpCatID.alias("ExpensesCategoryCode"),  
            target_table_arotron.ExpCatID.cast("string").alias("ExpensesCategory"),                
            target_table_arotron.ExpStatusID.cast("string").alias("ExpensesStatusCode"),   
            target_table_arotron.ExpenseStatusName.alias("ExpensesStatus"),       
            target_table_arotron.VATPs.cast("string").alias("VATPercentage"),
            lit(None).cast("int").alias("PacketCode"),
            lit(None).cast("string").alias("Packet"),
            target_table_arotron.ExpComments.cast("string").alias("Comments"),
            lit(None).cast("int").alias("LawyerKey"),   # we did not have that. From the other systems we had this as int       and IT gave this as string, therefore it can not be used here  
            lit(None).cast("int").alias("ExternalCompanyPartyKey"), # same as LawyerKey
            target_table_arotron.InvoiceDate.cast("timestamp").alias("InvoiceDate"),    
            target_table_arotron.InvNumber.cast("string").alias("InvoiceNumber"),
            lit(None).cast("int").alias("InvoiceApprovalStatusCode"), # same as LawyerKey
            lit(None).cast("int").alias("InvoiceApprovalStatus"), # same as LawyerKey
            lit(None).cast("int").alias("ExpensesValidationStatusCode"),
            lit(None).cast("int").alias("ExpensesValidationStatus"),
            lit(None).cast("int").alias("ExpensesApprovalStatusCode"),  # same as LawyerKey
            lit(None).cast("int").alias("ExpensesApprovalStatus"),  # same as LawyerKey
            target_table_arotron.Businessdate.cast("date").alias("BusinessDate"),
            #coalesce(portfolio.Arotron_dec_flag, lit("0")).alias("Arotron_dec_flag"), # Legal Actions QC Decommission to Arotron - notebooks run
        ).distinct()

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We keep only the portfolios for which the decommission of LAs has happened
        #result_df_arotron = result_df_arotron.filter("Arotron_dec_flag == 1")

        # Legal Actions QC Decommission to Arotron - notebooks run
        # We drop the column "Arotron_dec_flag" from the final dataframe
        #r#esult_df_arotron = result_df_arotron.drop("Arotron_dec_flag")

        # ================== Append ==================

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                #destination_schema_zone, # change for testing
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_expenses_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix) # change for testing